In [ ]:
%pip install fpdf2 pandas


In [ ]:
dbutils.library.restartPython()


In [ ]:
import pandas as pd
from fpdf import FPDF
from datetime import datetime

df = pd.read_csv("/Volumes/insight/default/titanic/Titanic.csv")

print(f"✅ Data loaded — {df.shape[0]} rows x {df.shape[1]} columns")


In [ ]:
def get_basic_summary(df):
    return {
        "rows"                : df.shape[0],
        "columns"             : df.shape[1],
        "column_names"        : list(df.columns),
        "dtypes"              : df.dtypes.astype(str).to_dict(),
        "missing_values"      : df.isnull().sum().to_dict(),
        "missing_percent"     : (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicates"          : int(df.duplicated().sum()),
        "numeric_columns"     : list(df.select_dtypes(include="number").columns),
        "categorical_columns" : list(df.select_dtypes(include="object").columns),
        "total_missing_cells" : int(df.isnull().sum().sum()),
        "memory_usage_kb"     : round(df.memory_usage(deep=True).sum() / 1024, 2),
    }

summary = get_basic_summary(df)
print("✅ Summary ready")


In [ ]:
def generate_pdf_report(df, summary, insights, filename="insightforge_report.pdf"):
    """
    Generates a professional PDF report with:
    - Dataset overview
    - Missing values table
    - Statistical summary
    - AI generated insights
    Saves to DBFS FileStore.
    """

    class InsightReport(FPDF):
        def header(self):
            self.set_font("Helvetica", "B", 15)
            self.set_fill_color(41, 128, 185)
            self.set_text_color(255, 255, 255)
            self.cell(0, 12, "InsightForge AI - Data Analysis Report",
                      align="C", fill=True,
                      new_x="LMARGIN", new_y="NEXT")
            self.set_text_color(0, 0, 0)
            self.set_font("Helvetica", "", 9)
            self.cell(0, 6,
                      f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                      align="C", new_x="LMARGIN", new_y="NEXT")
            self.ln(4)

        def footer(self):
            self.set_y(-15)
            self.set_font("Helvetica", "I", 8)
            self.set_text_color(128, 128, 128)
            self.cell(0, 10, f"InsightForge AI  |  Page {self.page_no()}",
                      align="C")

        def section_title(self, title):
            self.set_font("Helvetica", "B", 12)
            self.set_fill_color(235, 245, 255)
            self.cell(0, 8, title, fill=True,
                      new_x="LMARGIN", new_y="NEXT")
            self.ln(2)

    # ── Create PDF ────────────────────────────────────────────
    pdf = InsightReport()
    pdf.add_page()

    # ── Section 1: Dataset Overview ───────────────────────────
    pdf.section_title("1.  Dataset Overview")
    pdf.set_font("Helvetica", "", 11)
    pdf.cell(0, 7, f"  Rows              : {summary['rows']}",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Columns           : {summary['columns']}",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Duplicate rows    : {summary['duplicates']}",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Total missing     : {summary['total_missing_cells']} cells",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Memory usage      : {summary['memory_usage_kb']} KB",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Numeric columns   : {', '.join(summary['numeric_columns'])}",
             new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"  Category columns  : {', '.join(summary['categorical_columns'])}",
             new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)

    # ── Section 2: Missing Values ─────────────────────────────
    pdf.section_title("2.  Missing Values")
    pdf.set_font("Helvetica", "", 11)

    has_missing = False
    for col, count in summary["missing_values"].items():
        if count > 0:
            pct = summary["missing_percent"][col]
            pdf.cell(0, 7, f"  {col:20} : {count} missing  ({pct}%)",
                     new_x="LMARGIN", new_y="NEXT")
            has_missing = True

    if not has_missing:
        pdf.cell(0, 7, "  No missing values found — dataset is complete!",
                 new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)

    # ── Section 3: Statistical Summary ───────────────────────
    pdf.section_title("3.  Statistical Summary")
    pdf.set_font("Helvetica", "", 9)

    stats_df  = df.describe().round(2)
    col_count = len(stats_df.columns)
    col_w     = 160 / (col_count + 1)

    # Table header
    pdf.set_font("Helvetica", "B", 9)
    pdf.set_fill_color(200, 220, 255)
    pdf.cell(col_w, 6, "Stat", border=1, fill=True)
    for col in stats_df.columns:
        pdf.cell(col_w, 6, str(col)[:10], border=1, fill=True)
    pdf.ln()

    # Table rows
    pdf.set_font("Helvetica", "", 8)
    for idx in stats_df.index:
        pdf.cell(col_w, 5, str(idx), border=1)
        for col in stats_df.columns:
            pdf.cell(col_w, 5, str(stats_df.loc[idx, col]), border=1)
        pdf.ln()
    pdf.ln(6)

    # ── Section 4: AI Insights ────────────────────────────────
    pdf.add_page()
    pdf.section_title("4.  AI Generated Insights  (Powered by Gemini)")
    pdf.set_font("Helvetica", "", 10)

    # Clean markdown symbols for PDF
    clean = (insights
             .replace("**", "")
             .replace("##", "")
             .replace("#",  "")
             .replace("*",  "-"))

    pdf.multi_cell(0, 5, clean)

    # ── Save PDF ──────────────────────────────────────────────
    # Write directly to Unity Catalog Volume (serverless-compatible)
    volume_path = f"/Volumes/insight/default/titanic/{filename}"
    pdf.output(volume_path)

    print(f"✅ PDF saved to : {volume_path}")
    print(f"   Pages        : {pdf.page_no()}")
    return volume_path

print("✅ generate_pdf_report() defined")


In [ ]:

insights = """
Sending data to Gemini AI...

Here is an executive-level analysis of the dataset, structured for data engineering, analytical, and business decision-making purposes.

---

### 1. Dataset Overview

This dataset contains **1,309 records** representing the complete passenger manifest from the **Titanic passenger survival cohort**. It comprises **28 variables**, recording demographic details, ticketing socio-economic status, cabin/family groupings, boarding ports, and survival outcomes.

* **Primary Key:** `Passengerid` (1 to 1309)
* **Target Variable:** `2urvived` (Binary: 0 = Did not survive, 1 = Survived)
* **Key Features:** `Age`, `Fare`, `Sex` (1 = Female, 0 = Male), `Pclass` (Passenger Class: 1, 2, 3), `sibsp` (Siblings/Spouses onboard), `Parch` (Parents/Children onboard), `Embarked` (Port of Embarkation).
* **Data Hygiene State:** Uncleaned raw extract. Includes **19 completely empty/redundant zero-value columns** (`zero` through `zero.18`) created during prior ETL processes.

---

### 2. Key Findings

1. **Overall Survival Baseline (26.1%):** 
   Across the 1,309 passenger records, the mean survival rate is **0.261** (26.1%). 
2. **Gender Ratio Skew (35.6% Female):** 
   Female passengers (`Sex = 1`) represent **35.6%** of the dataset (mean = 0.356), while male passengers (`Sex = 0`) constitute **64.4%**.
3. **Socio-Economic Distribution Skew:** 
   * More than **50% of passengers were in 3rd Class** (`Pclass` median = 3.0, mean = 2.29).
   * Ticket pricing (`Fare`) exhibits extreme right skewness: while the median fare is **$14.45**, the mean is **$33.28**, and the maximum fare reaches **$512.33**. Additionally, minimum fare is **$0.00** (indicating complimentary/crew/working tickets).
4. **Demographic Profile:** 
   The average passenger age is **29.5 years** (std = 12.91), ranging from infants (**0.17 years / ~2 months**) up to seniors (**80.0 years**). The 25th to 75th percentile spans **22 to 35 years old**.
5. **Severe Schema Inefficiency (67.8% Redundant Columns):** 
   Out of 28 columns, **19 columns contain exclusively zero values** (`mean = 0.0, std = 0.0, min/max = 0.0`). These represent 67.8% of the column schema space.

---

### 3. Business Insights & Recommendations

*Contextualizing survival analysis into risk management, customer segmentation, and operational safety standards:*

* **Insight 1: High-Value Customer (1st Class) Protection vs. Mass Market Exposure**
  * *Observation:* The concentration of passengers in 3rd class (Pclass median 3) combined with high fare variance highlights a heavy volume base paying low fares ($7.90-$14.45) alongside a high-margin VIP segment paying up to $512.33.
  * *Recommendation:* Modern passenger transport and hospitality operations must ensure that emergency/crisis protocols treat capacity democratically, mitigating structural risks where lower-tier ticket holders face disproportionate operational delays or safety hazards.

* **Insight 2: Family Unit Risk & Group Dynamics**
  * *Observation:* Most passengers traveled alone (`sibsp` and `Parch` 75th percentiles are 1 and 0 respectively), but maximum family sizes reach up to **8 siblings/spouses** and **9 parents/children**.
  * *Recommendation:* Emergency evacuation procedures and ticketing systems should explicitly link group manifests. Operational response plans perform worse when families attempt to locate each other during crises; automated group check-in/tracking improves safety compliance and throughput.

* **Insight 3: Unmonetized/Complementary Passes ($0 Fares)**
  * *Observation:* At least 25% of fares were below $7.90, and the minimum fare recorded is $0.00.
  * *Recommendation:* Audit non-revenue passengers (complimentary, employee, or promotional tickets) to quantify operational yield loss and ensure non-paying passengers do not dilute high-fare cabin capacity during peak demand.

---

### 4. Data Quality Issues & Remediation Plan

| Issue | Severity | Description | Remediation Step |
| :--- | :--- | :--- | :--- |
| **Redundant Columns** | **High** | 19 columns (`zero` to `zero.18`) contain no data variance (100% zero values). | Drop all 19 `zero` columns to reduce schema bloat: <br>`df.drop(columns=[c for c in df if 'zero' in c], inplace=True)` |
| **Corrupted Target Name** | **Medium** | Target column `2urvived` starts with a digit/typo. | Rename column: <br>`df.rename(columns={'2urvived': 'Survived'}, inplace=True)` |
| **Missing Values** | **Low** | `Embarked` feature has 2 missing values. | Impute using mode (most common port): <br>`df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)` |
| **Suboptimal Data Types** | **Low** | Categorical variables (`Sex`, `Pclass`, `Embarked`, `Survived`) are stored as float64/int64. | Cast numerical flags to explicit categorical/boolean types to reduce memory overhead and improve analytical model clarity. |
| **Zero-Fare Anomalies** | **Low** | Multiple records contain `Fare == 0.0`. | Verify whether $0 fares represent crew, missing values, or complimentary passes; flag via binary indicator `IsFreeTicket`. |

---

### 5. Suggested Next Steps

1. **Data Pipeline Cleanup:**
   Execute a cleaning script to prune the 19 zero-columns, rename `2urvived` -> `Survived`, encode `Sex` (0/1 to Male/Female labels), and map `Embarked` codes back to port names (C, Q, S).

2. **Cross-Tabulation & Survival Driver Analysis (EDA):**
   * **Gender vs. Survival:** Calculate exact survival rates segmented by `Sex`.
   * **Class vs. Survival:** Measure survival proportion by `Pclass` (1st vs 2nd vs 3rd).
   * **Interaction Effect:** Perform a 3-way analysis (`Sex` x `Pclass` x `Survived`) to evaluate the "women and children first" rule across economic classes.

3. **Feature Engineering:**
   * **`FamilySize`**: Combine `sibsp + Parch + 1` to create total family count.
   * **`IsAlone`**: Binary flag (`1` if `FamilySize == 1`, else `0`).
   * **`AgeGroup`**: Bin continuous age into discrete buckets: `Child` (<12), `Teens` (12-18), `Adult` (19-59), `Senior` (60+).

4. **Predictive Modeling:**
   Develop a Logistic Regression or Random Forest classifier to predict survival probability, generating Feature Importance scores to statistically quantify the impact of Ticket Class, Fare, Age, and Gender on survival outcomes.
"""

# Generate the PDF
path = generate_pdf_report(df, summary, insights)
print(f"\n✅ Report generated at: {path}")


In [ ]:
# Create a download link
from IPython.display import HTML

download_link = f'<a href="/files/insightforge_report.pdf" target="_blank">⬇️ Click here to download your PDF report</a>'
HTML(download_link)


In [ ]:
# Check the file exists in Unity Catalog Volume
files = dbutils.fs.ls("/Volumes/insight/default/titanic/")
for f in files:
    if "report" in f.name:
        size_kb = round(f.size / 1024, 2)
        print(f"✅ Found: {f.name}  —  {size_kb} KB")
